# Diabetes Risk Prediction: Comparative Machine Learning and Model Interpretation

## Research framing

This project investigates whether routinely collected demographic and physiological measurements can predict diabetes status.

The analysis focuses on **reproducible machine-learning methodology** rather than clinical deployment.

### Research questions

1. How well can diabetes status be predicted from the available measurements?
2. How do interpretable and nonlinear models compare?
3. Which evaluation metrics are most informative for this imbalanced binary classification problem?
4. Which variables contribute most to predictive performance?
5. How does handling physiologically implausible zero values affect the modeling workflow?

### Models

- Logistic Regression — interpretable baseline
- Linear SVM
- RBF-kernel SVM
- Random Forest

### Evaluation

- Stratified 5-fold cross-validation
- Accuracy
- Precision
- Recall
- F1-score
- ROC-AUC
- confusion matrix
- ROC curves
- permutation importance

**Important:** This is an educational/research model, not a clinically validated diagnostic tool.


## 1. Imports and reproducibility

In [ ]:
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    classification_report, confusion_matrix, roc_curve
)
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

PROJECT_DIR = Path.cwd().resolve()
DATA_PATH = PROJECT_DIR / "data" / "diabetes.csv"

if not DATA_PATH.exists():
    DATA_PATH = PROJECT_DIR.parent / "data" / "diabetes.csv"

RESULTS_DIR = PROJECT_DIR / "results"
FIGURES_DIR = RESULTS_DIR / "figures"

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

print("Data:", DATA_PATH)
print("Results:", RESULTS_DIR)
print("Figures:", FIGURES_DIR)


## 2. Load and inspect the dataset

In [ ]:
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
display(df.describe().T)

print("\nMissing values:")
display(df.isna().sum())

print("\nClass distribution:")
display(df["Outcome"].value_counts())
print("\nClass proportions:")
display(df["Outcome"].value_counts(normalize=True).round(3))


## 3. Data quality: physiologically implausible zeros

In this dataset, zero values in several medical measurements are generally treated as missing rather than as true physiological measurements.

Affected variables:

- Glucose
- BloodPressure
- SkinThickness
- Insulin
- BMI

We preserve the original dataset and create a modeling copy where these zeros are replaced with missing values. The imputation is then performed **inside the model pipeline**, ensuring that imputation statistics are learned only from training data.


In [ ]:
zero_as_missing = [
    "Glucose",
    "BloodPressure",
    "SkinThickness",
    "Insulin",
    "BMI"
]

zero_summary = pd.DataFrame({
    "zero_count": [(df[c] == 0).sum() for c in zero_as_missing],
    "zero_percent": [100 * (df[c] == 0).mean() for c in zero_as_missing]
}, index=zero_as_missing)

display(zero_summary.round(2))

model_df = df.copy()

for col in zero_as_missing:
    model_df[col] = model_df[col].replace(0, np.nan)

print("Missing values after zero-as-missing treatment:")
display(model_df.isna().sum())


## 4. Exploratory data analysis

In [ ]:
# Outcome distribution
plt.figure(figsize=(7, 5))
plt.hist(df["Outcome"], bins=[-0.5, 0.5, 1.5], rwidth=0.8)
plt.xticks([0, 1], ["No Diabetes", "Diabetes"])
plt.title("Diabetes Outcome Distribution")
plt.xlabel("Outcome")
plt.ylabel("Number of observations")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "outcome_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

# Feature distributions by outcome
features = [c for c in model_df.columns if c != "Outcome"]

fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, feature in zip(axes.flat, features):
    groups = [
        model_df.loc[model_df["Outcome"] == 0, feature].dropna(),
        model_df.loc[model_df["Outcome"] == 1, feature].dropna()
    ]
    ax.boxplot(groups, labels=["No Diabetes", "Diabetes"])
    ax.set_title(feature)
    ax.tick_params(axis="x", rotation=20)

plt.suptitle("Feature Distributions by Diabetes Outcome", y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / "features_by_outcome.png", dpi=300, bbox_inches="tight")
plt.show()


In [ ]:
plt.figure(figsize=(10, 8))
corr = model_df.corr(numeric_only=True)

sns.heatmap(
    corr,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0
)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "correlation_heatmap.png", dpi=300, bbox_inches="tight")
plt.show()


## 5. Train/test split

A **stratified** split is used so that the proportion of diabetes-positive observations is approximately preserved in both sets.

All preprocessing is performed inside pipelines to prevent test-set information from influencing training transformations.


In [ ]:
X = model_df.drop(columns="Outcome")
y = model_df["Outcome"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    stratify=y,
    random_state=RANDOM_STATE
)

print("Training size:", X_train.shape)
print("Test size:", X_test.shape)
print("\nTraining class distribution:")
print(y_train.value_counts(normalize=True).round(3))
print("\nTest class distribution:")
print(y_test.value_counts(normalize=True).round(3))


## 6. Model definitions

Four models are compared.

- **Logistic Regression:** interpretable linear baseline.
- **Linear SVM:** margin-based linear classifier.
- **RBF SVM:** nonlinear kernel model.
- **Random Forest:** tree-based nonlinear ensemble.

Median imputation and standardization are part of the pipeline. The Random Forest does not require scaling, but keeping its preprocessing separate makes the experiment explicit.


In [ ]:
linear_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))
])

linear_svm_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="linear", probability=True, random_state=RANDOM_STATE))
])

rbf_svm_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", SVC(kernel="rbf", probability=True, random_state=RANDOM_STATE))
])

rf_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("model", RandomForestClassifier(
        n_estimators=500,
        min_samples_leaf=3,
        class_weight="balanced",
        random_state=RANDOM_STATE,
        n_jobs=-1
    ))
])

models = {
    "Logistic Regression": linear_pipeline,
    "Linear SVM": linear_svm_pipeline,
    "RBF SVM": rbf_svm_pipeline,
    "Random Forest": rf_pipeline
}


## 7. Stratified 5-fold cross-validation

In [ ]:
cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "ROC_AUC": "roc_auc"
}

cv_rows = []

for name, pipeline in models.items():
    scores = cross_validate(
        pipeline,
        X_train,
        y_train,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    cv_rows.append({
        "Model": name,
        "CV_Accuracy": scores["test_Accuracy"].mean(),
        "CV_Precision": scores["test_Precision"].mean(),
        "CV_Recall": scores["test_Recall"].mean(),
        "CV_F1": scores["test_F1"].mean(),
        "CV_ROC_AUC": scores["test_ROC_AUC"].mean()
    })

cv_results = pd.DataFrame(cv_rows).sort_values("CV_ROC_AUC", ascending=False)
display(cv_results.round(4))

cv_results.to_csv(RESULTS_DIR / "cross_validation_results.csv", index=False)


## 8. Final held-out test evaluation

In [ ]:
test_rows = []
test_predictions = {}
test_probabilities = {}

for name, pipeline in models.items():
    pipeline.fit(X_train, y_train)

    pred = pipeline.predict(X_test)
    prob = pipeline.predict_proba(X_test)[:, 1]

    test_predictions[name] = pred
    test_probabilities[name] = prob

    test_rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
        "ROC_AUC": roc_auc_score(y_test, prob)
    })

test_results = pd.DataFrame(test_rows).sort_values("ROC_AUC", ascending=False)

display(test_results.round(4))
test_results.to_csv(RESULTS_DIR / "test_results.csv", index=False)


## 9. Classification reports and confusion matrices

In [ ]:
best_model_name = test_results.iloc[0]["Model"]
print("Best model by held-out ROC-AUC:", best_model_name)

for name, pred in test_predictions.items():
    print("\n" + "=" * 70)
    print(name)
    print("=" * 70)
    print(classification_report(y_test, pred, zero_division=0))

    cm = confusion_matrix(y_test, pred)

    plt.figure(figsize=(5, 4))
    sns.heatmap(
        cm,
        annot=True,
        fmt="d",
        cmap="Blues",
        xticklabels=["No Diabetes", "Diabetes"],
        yticklabels=["No Diabetes", "Diabetes"]
    )
    plt.xlabel("Predicted")
    plt.ylabel("Actual")
    plt.title(f"Confusion Matrix — {name}")
    plt.tight_layout()

    safe_name = name.lower().replace(" ", "_")
    plt.savefig(
        FIGURES_DIR / f"confusion_matrix_{safe_name}.png",
        dpi=300,
        bbox_inches="tight"
    )
    plt.show()


## 10. ROC curve comparison

In [ ]:
plt.figure(figsize=(9, 7))

for name, probabilities in test_probabilities.items():
    fpr, tpr, _ = roc_curve(y_test, probabilities)
    auc = roc_auc_score(y_test, probabilities)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")

plt.plot([0, 1], [0, 1], linestyle="--", label="Random classifier")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve Comparison")
plt.legend()
plt.tight_layout()
plt.savefig(FIGURES_DIR / "roc_curve_comparison.png", dpi=300, bbox_inches="tight")
plt.show()


## 11. Permutation importance

Permutation importance measures how much predictive performance changes when each original feature is randomly shuffled.

This provides a model-agnostic interpretation of which measurements are most useful for prediction.


In [ ]:
best_pipeline = models[best_model_name]

perm = permutation_importance(
    best_pipeline,
    X_test,
    y_test,
    scoring="roc_auc",
    n_repeats=30,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance = (
    pd.Series(perm.importances_mean, index=X_test.columns)
    .sort_values(ascending=False)
)

importance_df = importance.to_frame("mean_decrease_in_ROC_AUC")
importance_df["std"] = perm.importances_std[
    np.argsort(perm.importances_mean)[::-1]
]

display(importance_df.head(10))

importance_df.to_csv(RESULTS_DIR / "permutation_importance.csv")

plt.figure(figsize=(9, 6))
importance.head(10).sort_values().plot(kind="barh")
plt.xlabel("Mean decrease in ROC-AUC")
plt.title(f"Top Predictive Features — {best_model_name}")
plt.tight_layout()
plt.savefig(FIGURES_DIR / "permutation_importance.png", dpi=300, bbox_inches="tight")
plt.show()


## 12. Interpretation and limitations

### Interpretation

The models estimate predictive relationships between measured characteristics and diabetes status. Predictive importance should **not** be interpreted as causal importance.

Because this dataset is observational, the analysis cannot establish that changing any particular variable would cause a change in diabetes risk.

### Important limitations

1. The dataset is relatively small.
2. The observations come from a specific population and may not generalize to other groups.
3. Several medical measurements contain zero values that are treated as missing based on domain plausibility.
4. The model is not externally validated.
5. The held-out test set is only one sample; cross-validation is therefore reported alongside final test performance.
6. The dataset and models are not sufficient for clinical diagnosis.

### Future research

- external validation on an independent cohort
- probability calibration
- threshold optimization based on clinical costs
- subgroup/fairness analysis
- confidence intervals through repeated resampling
- explainable AI methods such as SHAP
- comparison with clinically validated risk scores


## 13. Reproducibility

The experiment uses:

- fixed random seed (`42`)
- stratified train/test split
- stratified 5-fold cross-validation
- preprocessing pipelines
- saved metrics
- saved figures

Install dependencies:

```bash
pip install -r requirements.txt
```

Then run the notebook from the repository root.

All generated artifacts are written to:

```text
results/
├── cross_validation_results.csv
├── test_results.csv
├── permutation_importance.csv
└── figures/
```
